# Arquitectura backend + base de experimentación (issue #12)
Usa los **mismos casos de uso** que la API (`backend/src/collection/`).
Backend **solo-arbol por decisión**: dataset, inferencia con el árbol ajustado y planificación de rutas. La comparación de modelos vive en los notebooks, no en el backend.
Este notebook verifica estructura y fixtures; no entrena ni compara (eso vive en el notebook 10).

In [ ]:
import sys
from pathlib import Path

# Funciona desde la raíz del repo y desde notebooks/ (nbval ejecuta
# el kernel con cwd = directorio del notebook).
CANDIDATOS = [Path.cwd() / "backend" / "src",
              Path.cwd().parent / "backend" / "src"]
SRC = next(c for c in CANDIDATOS if (c / "collection").is_dir())
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import collection
print("src:", SRC.exists(), "schema:", collection.SCHEMA_VERSION)

## Parámetros (el único lugar que hay que editar)

In [ ]:
# Cambiar dataset SOLO acá.
# SRC viene de la celda de setup; el default apunta al fixture local.
DATASET_DIR = str((SRC.parent / "tests" / "fixtures") if (SRC.parent / "tests" / "fixtures").is_dir() else (SRC / ".." / "tests" / "fixtures"))
print("dataset:", DATASET_DIR)

## 1. Estructura pelada (contratos abstractos, sin lógica)

In [ ]:
from abc import ABC

from collection.adapters.datasets.csv_dataset_reader import CsvDatasetReader
from collection.adapters.graphs.networkx_route_planner import NetworkxRoutePlanner
from collection.adapters.models.sklearn_predictor import SklearnPredictor
from collection.application.use_cases.plan_route.contract import PlanRouteContract
from collection.application.use_cases.predict_priorities.contract import PredictPrioritiesContract
from collection.application.use_cases.validate_load_dataset.contract import ValidateLoadDatasetContract
from collection.domain.entities.node import Node
from collection.domain.entities.predicted_segment import PredictedSegment
from collection.domain.entities.route import Route
from collection.domain.entities.route_config import RouteConfig
from collection.domain.entities.segment_features import SegmentFeatures

assert collection.SCHEMA_VERSION == "v1", "schema inesperado"
for contract in (ValidateLoadDatasetContract, PredictPrioritiesContract, PlanRouteContract):
    assert issubclass(contract, ABC), contract
    assert "execute" in contract.__abstractmethods__, contract

assert CsvDatasetReader is not None
assert NetworkxRoutePlanner is not None
assert SklearnPredictor is not None


## 2. Fixtures locales (esquema v1, sin pandas)

In [ ]:
import csv
from pathlib import Path

base = Path(DATASET_DIR)
nodos_p = base / "nodos.csv"
tramos_p = base / "tramos.csv"
assert nodos_p.is_file(), f"falta {nodos_p}"
assert tramos_p.is_file(), f"falta {tramos_p}"

with open(nodos_p, newline="", encoding="utf-8") as f:
    nodos = list(csv.DictReader(f))
with open(tramos_p, newline="", encoding="utf-8") as f:
    tramos = list(csv.DictReader(f))

print(f"nodos={len(nodos)} tramos={len(tramos)}")
assert len(nodos) == 6, f"fixture nodos cambió: {len(nodos)}"
assert len(tramos) == 12, f"fixture tramos cambió: {len(tramos)}"
assert {r["prioridad_recoleccion"] for r in tramos} <= {"baja", "media", "alta"}
print("ejemplo:", tramos[0]["tramo_id"], tramos[0]["prioridad_recoleccion"])

## 3. Composición pelada (los stubs levantan NotImplementedError por diseño)

In [ ]:
from collection import composition

# La comparacion no se cablea en el backend: vive en los notebooks.
assert not hasattr(composition, "create_compare_models_use_case"), "el backend no debe exponer comparacion"

for nombre in ("create_validate_load_dataset_use_case",):
    caso = getattr(composition, nombre)()
    print(nombre, "-> implementado:", type(caso).__name__)

for nombre in ("create_plan_route_use_case",):
    caso = getattr(composition, nombre)()
    print(nombre, "-> implementado:", type(caso).__name__)

try:
    composition.create_predict_priorities_use_case("bosque-magico", pipeline=object())
except ValueError as exc:
    print("predict rechaza modelo desconocido:", exc)
else:
    raise AssertionError("predict debería rechazar modelos desconocidos")

# Reglas de dominio también son plantillas (S8):
from collection.domain.rules.label_v1 import label_v1
try:
    label_v1(2, 0, 1, "residential")
except NotImplementedError:
    print("label_v1 -> NotImplementedError (esperado, pendiente)")
else:
    raise AssertionError("label_v1 debería ser plantilla")


> **Aviso:** la etiqueta es *construida* con la regla v1: el modelo aprenderá esa regla y una buena métrica **no** demostrará utilidad operativa real. La comparación/entrenamiento vive en el notebook 10 sobre la misma preparación del backend; el backend solo sirve el árbol ya elegido.